In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [25]:
fp = "tmp0kekil41.csv"
df = pd.read_csv(fp)


/var/folders/pc/7dj9yy6d0yj9lhpgvnlq86wr0000gp/T/ipykernel_4750/516374522.py:2: DtypeWarning:

Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.



In [26]:
#print(df.head())
print(df.iloc[1])
violation_counts = df["violdesc"].value_counts()
#print(violation_counts)
violation_levels = df["viol_level"].value_counts()
#print(violation_levels)


businessname                                   1000 Degrees Pizza
dbaname                                                       NaN
legalowner                                           KHOSLA VIPAN
namelast                                          Pasquriello LLC
namefirst                                    Kenneth Pasquariello
licenseno                                                  313440
issdttm                                    2017-08-14 12:49:37+00
expdttm                                    2020-01-01 04:59:00+00
licstatus                                                Inactive
licensecat                                                     FS
descript                                        Eating & Drinking
result                                                    HE_Fail
resultdttm                                 2018-03-20 14:54:25+00
violation                                         22-4-601/602.11
viol_level                                                     **
violdesc  

In [34]:
df = df[df["licensecat"] == "FS"]


df = df.drop_duplicates()
df["violdttm"] = pd.to_datetime(df["violdttm"], errors = "coerce")
df["issdttm"] = pd.to_datetime(df["issdttm"], errors = "coerce")
df["expdttm"] = pd.to_datetime(df["expdttm"], errors = "coerce")

df["violdesc"] = df["violdesc"].str.strip()
df["businessname"] = df["businessname"].str.strip()


#Zip code cleaning
df = df.dropna(subset=['zip'])
df['zip'] = df['zip'].astype(str)
df['zip'] = df['zip'].str.split('.').str[0]
df = df[df['zip'].str.isnumeric()]
df['zip'] = df['zip'].str.zfill(5)
print(df['zip'].unique())
print("Number of unique ZIP codes:", df['zip'].nunique())


['02108' '02118' '02110' '02131' '02125' '02116' '02114' '02135' '02129'
 '02111' '02134' '02109' '02128' '02132' '02115' '02122' '02215' '02199'
 '02210' '02130' '02127' '02119' '02113' '02136' '02188' '02120' '02124'
 '02126' '02201' '02121' '02163' '02446' '02467']
Number of unique ZIP codes: 33


In [35]:
df_fail = df[df["viol_status"] == "Fail"]
main_levels = ["*", "**", "***"]
df_fail = df_fail[df_fail["viol_level"].isin(main_levels)]

df_fail['year'] = df_fail['violdttm'].dt.year
print(df_fail['zip'].unique())
viol_business_count = df_fail.groupby(['businessname','year','viol_level']).size().reset_index(name='num_violations')
print(viol_business_count.head(10))
viol_zip_count = df_fail.groupby(['zip','year','viol_level']).size().reset_index(name='num_violations')
print(viol_zip_count.head(10))


['02108' '02118' '02131' '02125' '02116' '02114' '02135' '02129' '02111'
 '02110' '02134' '02109' '02128' '02132' '02115' '02122' '02215' '02199'
 '02210' '02130' '02127' '02119' '02113' '02136' '02188' '02120' '02124'
 '02126' '02201' '02121' '02163' '02446' '02467']
               businessname  year viol_level  num_violations
0  100 Percent Delicia Food  2013          *              16
1  100 Percent Delicia Food  2013        ***               1
2  100 Percent Delicia Food  2014          *               3
3  100 Percent Delicia Food  2015          *              13
4  100 Percent Delicia Food  2015        ***               1
5  100 Percent Delicia Food  2016          *              18
6  100 Percent Delicia Food  2016        ***               5
7  100 Percent Delicia Food  2017          *              19
8  100 Percent Delicia Food  2017        ***               3
9  100 Percent Delicia Food  2018          *              11
     zip  year viol_level  num_violations
0  02108  2007    

In [45]:


fig = px.line(
    viol_zip_count, 
    x = 'year', 
    y = 'num_violations', 
    color = 'viol_level',
    line_group = 'zip',
    hover_name = 'zip',
    markers = True,
    title = 'Timeline of Violations by ZIP Code (2006-2026)'
)
trace = (
    viol_zip_count[['zip', 'viol_level']]
    .drop_duplicates()
    .sort_values(['viol_level', 'zip'])
)

zips = sorted(viol_zip_count['zip'].unique())
zip_buttons = []
zip_buttons.append(dict(
    label = "All ZIPs",
    method = 'update',
    args =[
        {'visible': [True] * len(fig.data)}, 
        {'title': "Violations Over Time for All ZIPs"}
    ]


))

for z in zips:
    zip_buttons.append(dict(
        label = str(z),
        method = "update",
        args = [
            {'visible': [row.zip == str(z) for row in trace.itertuples()]},
            {"title": f"Violations Over Time for ZIP {z}"}
        ]
    ))

fig.update_layout(
    updatemenus = [
        dict(
            buttons = [
                dict(
                    label = "All ZIPs",
                    method = 'update',
                    args = [{"visible": [True]*len(fig.data)},
                            {"title": "Violations Over Time for All ZIPs"}]
                )
            ] + zip_buttons,
            direction = 'down',
            showactive = True
        )
    ]
)

fig.update_layout(
    xaxis = dict(dtick = 1),
    yaxis_title = "Number of Violations",
    legend_title = "Violation Level",
    hovermode = 'x unified',
    title_x = 0.5,
    legend = dict(
        title = 'Violation Level',
        orientation = 'v',
        x = 1.02,
        y = 1
    )
)
fig.show()



In [46]:
fig.write_html('YehS.graphHomework7.html')